In [ ]:
import pandas as pd

r_cols = ['anime_id','name','genre','type','episodes','rating','members']
ratings = pd.read_csv('/content/anime.csv', sep=',', usecols=range(7), encoding="ISO-8859-1")

m_cols = ['user_id','anime_id','rating']
animes = pd.read_csv('/content/rating.csv', sep=',', usecols=m_cols, encoding="ISO-8859-1")

# Ensure 'anime_id' columns are of integer type to prevent merge issues
animes['anime_id'] = pd.to_numeric(animes['anime_id'], errors='coerce').fillna(0).astype(int)
ratings['anime_id'] = pd.to_numeric(ratings['anime_id'], errors='coerce').fillna(0).astype(int)

ratings = pd.merge(animes, ratings, on='anime_id')

# Clean column names by stripping whitespace
ratings.columns = ratings.columns.str.strip()

# Rename the merged rating columns
ratings = ratings.rename(columns={'rating_x': 'user_rating', 'rating_y': 'total_rating'})

In [ ]:
ratings.head()

,user_id,anime_id,user_rating,name,genre,type,episodes,total_rating,members
0,1,20,-1,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1,1,24,-1,School Rumble,"Comedy, Romance, School, Shounen",TV,26,8.06,178553
2,1,79,-1,Shuffle!,"Comedy, Drama, Ecchi, Fantasy, Harem, Magic, R...",TV,24,7.31,158772
3,1,226,-1,Elfen Lied,"Action, Drama, Horror, Psychological, Romance,...",TV,13,7.85,623511
4,1,241,-1,Girls Bravo: First Season,"Comedy, Ecchi, Fantasy, Harem, Romance, School",TV,11,6.69,84395


### Handling Missing User Ratings (Revisited)

Before creating a sparse matrix, it's essential to properly handle the `-1` values in the `user_rating` column. As discussed, these represent unrated items. Converting them to `NaN` ensures they are not treated as valid ratings and can be implicitly ignored when constructing the sparse matrix.

In [ ]:
import numpy as np

# Convert -1 in 'user_rating' to NaN
ratings['user_rating'] = ratings['user_rating'].replace(-1, np.nan)

# Display the head to show the change and ensure NaNs are present
display(ratings.head())

,user_id,anime_id,user_rating,name,genre,type,episodes,total_rating,members
0,1,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1,1,24,NaN,School Rumble,"Comedy, Romance, School, Shounen",TV,26,8.06,178553
2,1,79,NaN,Shuffle!,"Comedy, Drama, Ecchi, Fantasy, Harem, Magic, R...",TV,24,7.31,158772
3,1,226,NaN,Elfen Lied,"Action, Drama, Horror, Psychological, Romance,...",TV,13,7.85,623511
4,1,241,NaN,Girls Bravo: First Season,"Comedy, Ecchi, Fantasy, Harem, Romance, School",TV,11,6.69,84395


### Directly Creating a Sparse User-Item Matrix

To avoid the 'out of memory' error, we will directly construct a `scipy.sparse.csr_matrix` from the `ratings` DataFrame. This approach is highly memory-efficient for sparse data by only storing the non-zero (i.e., non-`NaN` ratings) elements. We will also create mappings to easily convert between original user IDs/anime names and their integer indices in the sparse matrix.

In [ ]:
import scipy.sparse as sp

# Filter out NaN ratings before creating the sparse matrix
# This ensures only actual ratings are included
valid_ratings = ratings.dropna(subset=['user_rating'])

# Create mappings for user_id and anime 'name' to integer indices
users = valid_ratings['user_id'].unique()
animes = valid_ratings['name'].unique()

user_to_idx = {user: i for i, user in enumerate(users)}
anime_to_idx = {anime: i for i, anime in enumerate(animes)}
idx_to_anime = {i: anime for i, anime in enumerate(animes)} # For easy lookup later

# Prepare data for sparse matrix
row_indices = valid_ratings['user_id'].map(user_to_idx).values
col_indices = valid_ratings['name'].map(anime_to_idx).values
data = valid_ratings['user_rating'].values

# Create the sparse matrix (CSR format is generally good for row-wise operations)
anime_user_sparse_matrix = sp.csr_matrix(
    (data, (row_indices, col_indices)),
    shape=(len(users), len(animes))
)

print(f"Shape of the sparse matrix: {anime_user_sparse_matrix.shape}")
print(f"Number of non-zero elements: {anime_user_sparse_matrix.nnz}")
print(f"Sparsity: {1 - anime_user_sparse_matrix.nnz / (anime_user_sparse_matrix.shape[0] * anime_user_sparse_matrix.shape[1]):.2%}")

Shape of the sparse matrix: (69600, 9926)
Number of non-zero elements: 6337232
Sparsity: 99.08%


### Creating a User-Item Interaction Matrix

With the `user_rating` column now representing individual user ratings (with unrated items as `NaN`), we can create a user-item interaction matrix using `pivot_table`. This matrix is a fundamental component for many recommendation system algorithms.

In [ ]:
# anime_user_sparse_matrix.head() # This method is not available for sparse matrices

# You can inspect its properties:
print(f"Shape of the sparse matrix: {anime_user_sparse_matrix.shape}")
print(f"Number of non-zero elements: {anime_user_sparse_matrix.nnz}")

# Or to view a small portion (use with caution for very large matrices):
display(anime_user_sparse_matrix[0:5, 0:5].toarray())

Shape of the sparse matrix: (69600, 9926)
Number of non-zero elements: 6337232


array([[10., 10., 10., 10.,  0.],
       [ 0.,  0.,  0.,  0., 10.],
       [ 6.,  0.,  9.,  0., 10.],
       [ 2.,  2.,  1.,  1.,  0.],
       [ 0.,  6.,  8.,  8.,  0.]])

In [ ]:
# Get the original user IDs and anime names for the first 5x5 slice
# We need the inverse mapping for user IDs
idx_to_user = {v: k for k, v in user_to_idx.items()}

sample_user_ids = [idx_to_user[i] for i in range(10)]
sample_anime_names = [idx_to_anime[i] for i in range(10)]

# Convert the 5x5 sparse matrix slice to a dense array and then to a DataFrame
sample_matrix_df = pd.DataFrame(
    anime_user_sparse_matrix[0:10, 0:10].toarray(),
    index=sample_user_ids,
    columns=sample_anime_names
)

display(sample_matrix_df)

,Highschool of the Dead,High School DxD,Sword Art Online,High School DxD New,Kuroko no Basket,Naruto,Shaman King,Slam Dunk,Sen to Chihiro no Kamikakushi,Dragon Ball GT
1,10.0,10.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0
3,6.0,0.0,9.0,0.0,10.0,8.0,6.0,9.0,10.0,9.0
5,2.0,2.0,1.0,1.0,0.0,6.0,0.0,6.0,8.0,1.0
7,0.0,6.0,8.0,8.0,0.0,0.0,0.0,9.0,0.0,7.0
8,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0
12,6.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0


In [ ]:
naruto_ratings = ratings[ratings['name'] == 'Naruto']
display(naruto_ratings.head())

,user_id,anime_id,user_rating,name,genre,type,episodes,total_rating,members
0,1,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
156,3,20,8.0,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
306,5,20,6.0,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
769,6,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1162,10,20,NaN,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297


### Building a K-Nearest Neighbors (KNN) Recommendation System

Now that we have our sparse user-item interaction matrix, we can use a K-Nearest Neighbors (KNN) algorithm to find similar items or users. We'll start with an item-based approach, meaning we'll find anime similar to a given anime based on how users have rated them.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# We'll use an item-based collaborative filtering approach.
# This means we want to find items (anime) that are similar to each other.
# For this, we need to transpose our sparse matrix so that anime are rows and users are columns.
# This allows NearestNeighbors to find neighbors among anime based on user ratings.

anime_features = anime_user_sparse_matrix.T

# Initialize the NearestNeighbors model.
# 'metric='cosine'' is often used for recommendation systems as it measures the angle between two vectors,
# which is good for determining similarity regardless of magnitude.
# 'algorithm='brute'' is simple but can be slow for very large datasets; alternatives like 'ball_tree' or 'kd_tree' can be faster.
# 'n_neighbors' specifies how many neighbors to find.
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

# Fit the model to our anime features (transposed sparse matrix)
knn_model.fit(anime_features)

print(f"KNN model fitted with {anime_features.shape[0]} items and {anime_features.shape[1]} features (users).")

KNN model fitted with 9926 items and 69600 features (users).


#### Example: Finding Similar Anime

Let's test our KNN model by finding anime similar to a particular anime. We'll pick 'Naruto' as an example, since you just looked at its ratings.

In [ ]:
# Function to get recommendations
def get_similar_animes(anime_name, model, data, anime_to_idx_map, idx_to_anime_map, n_recommendations=10):
    # Ensure the anime exists in our mapping
    if anime_name not in anime_to_idx_map:
        print(f"Anime '{anime_name}' not found in the dataset.")
        return []

    anime_idx = anime_to_idx_map[anime_name]
    # Get distances and indices of n_recommendations + 1 neighbors (first one is the anime itself)
    distances, indices = model.kneighbors(data[anime_idx], n_neighbors=n_recommendations + 1)

    # Flatten the arrays
    distances = distances.flatten()
    indices = indices.flatten()

    recommendations = []
    for i in range(len(distances)):
        # Skip the first result, as it's the anime itself
        if i == 0:
            continue
        recommendations.append({
            'anime': idx_to_anime_map[indices[i]],
            'similarity': 1 - distances[i] # Convert distance to similarity
        })

    # Sort by similarity in descending order
    recommendations = sorted(recommendations, key=lambda x: x['similarity'], reverse=True)

    return recommendations

# Get recommendations for 'Naruto'
naruto_recommendations = get_similar_animes(
    'Naruto',
    knn_model,
    anime_features,
    anime_to_idx,
    idx_to_anime
)

print(f"Recommendations for 'Naruto':")
for rec in naruto_recommendations:
    print(f"- {rec['anime']} (Similarity: {rec['similarity']:.4f})")

Recommendations for 'Naruto':
- Death Note (Similarity: 0.5518)
- Fullmetal Alchemist (Similarity: 0.4777)
- Bleach (Similarity: 0.4715)
- Fullmetal Alchemist: Brotherhood (Similarity: 0.4697)
- Code Geass: Hangyaku no Lelouch (Similarity: 0.4693)
- Sword Art Online (Similarity: 0.4692)
- Shingeki no Kyojin (Similarity: 0.4637)
- Dragon Ball Z (Similarity: 0.4574)
- Naruto Movie 1: Dai Katsugeki!! Yuki Hime Shinobu Houjou Dattebayo! (Similarity: 0.4536)
- Ao no Exorcist (Similarity: 0.4494)


### Accessing Public S3 Data from Colab

To read data directly from a public Amazon S3 bucket, we need to install the `s3fs` library. This library provides a Pythonic file system interface to S3, allowing `pandas` to open files stored there as if they were local.

In [1]:
# Install s3fs to enable pandas to read from S3
!pip install s3fs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 39.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.


This code block demonstrates how to load a CSV file from a public S3 bucket. If you have sensitive data, remember to keep your S3 buckets private and use authentication methods (like AWS credentials) for access.

### Securely Accessing Private S3 Buckets from Colab

Since making S3 buckets public carries risks, a more secure approach is to use AWS Identity and Access Management (IAM) user credentials. These credentials (Access Key ID and Secret Access Key) grant your Colab environment programmatic access to your private S3 bucket.

**Steps to set up IAM credentials:**

1.  **Create an IAM User in AWS:**
    *   Go to the AWS IAM Console.
    *   Navigate to **Users** and click **Add users**.
    *   Give the user a name (e.g., `colab-s3-access`).
    *   Select **Access key - Programmatic access** as the credential type.
    *   Proceed to **Permissions**. Attach a policy like `AmazonS3ReadOnlyAccess` for read-only access, or create a custom policy with `s3:GetObject` specifically for your bucket (`ml-data-bucket-tutorial-2026`).
    *   Complete the user creation. **Crucially, save the Access Key ID and Secret Access Key displayed on the final screen.** These will only be shown once.

2.  **Store Credentials in Colab Secrets:**
    *   In Google Colab, click the **'🔑 Secrets'** icon on the left sidebar.
    *   Click **'+ New secret'**.
    *   Add two secrets:
        *   **Name:** `AWS_ACCESS_KEY_ID`, **Value:** `your_aws_access_key_id` (from IAM)
        *   **Name:** `AWS_SECRET_ACCESS_KEY`, **Value:** `your_aws_secret_access_key` (from IAM)
    *   Ensure the 'Notebook access' toggle is enabled for these secrets.

3.  **Access S3 from Colab using `s3fs`:**
    *   The `s3fs` library (which `pandas` uses under the hood for S3) will automatically pick up AWS credentials from environment variables. We'll load them from Colab secrets and set them as environment variables.

In [ ]:
from google.colab import userdata
import os
import pandas as pd

# Retrieve credentials from Colab Secrets
# Ensure these secret names match what you set in the Colab Secrets panel
aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY')

# Set them as environment variables so s3fs can pick them up
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key

# Now, you can access your private S3 bucket.
# Replace with the actual S3 URI to your anime.csv file in your private bucket.
# For example, if your bucket is 'ml-data-bucket-tutorial-2026' and the file is 'anime_dataset/anime.csv'
private_s3_uri = 's3://ml-data-bucket-tutorial-2026/anime_dataset/anime.csv'

try:
    # Read the CSV directly from S3 using the configured credentials
    s3_private_df = pd.read_csv(private_s3_uri)
    print(f"Successfully loaded data from private S3 bucket: {private_s3_uri}")
    display(s3_private_df.head())
except Exception as e:
    print(f"Error loading data from private S3: {e}")
    print("Please ensure your IAM user has the correct permissions and the bucket is NOT public.")
